In [2]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_batch
import json
from datetime import datetime

import boto3
import time
from botocore.exceptions import ClientError


In [19]:

REGION = "eu-north-1"
SECRET_NAME = "arn:aws:secretsmanager:eu-north-1:637910750724:secret:Postgre-gGI3hV"

def get_secret():
    sm = boto3.client("secretsmanager", region_name=REGION)
    secret_str = sm.get_secret_value(SecretId=SECRET_NAME)["SecretString"]
    return json.loads(secret_str)

s = get_secret()

print(
    "HOST:", s["host"],
    "PORT:", s.get("port", 5432),
    "DB:", s.get("dbname"),
    "USER:", s["username"]
)

conn = psycopg2.connect(
    host=s["host"],
    port=s.get("port", 5432),
    dbname=s["dbname"],
    user=s["username"],
    password=s["password"],
    connect_timeout=10,
    sslmode="require"
)

print("OK conectado")
conn.close()


HOST: rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com PORT: 5432 DB: rawg-db USER: postgre


OperationalError: connection to server at "rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com" (51.21.189.147), port 5432 failed: timeout expired


In [14]:

REGION = "eu-north-1"
SECRET_NAME = "arn:aws:secretsmanager:eu-north-1:637910750724:secret:Postgre-gGI3hV"  # o ARN

def get_secret():
    sm = boto3.client("secretsmanager", region_name=REGION)
    s = sm.get_secret_value(SecretId=SECRET_NAME)["SecretString"]
    return json.loads(s)

s = get_secret()
print("HOST:", s["rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com"], "PORT:", s.get("port", 5432), "DB:", s["rawg-db"], "USER:", s["postgres"])

conn = psycopg2.connect(
    host=s["rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com"],
    port=s.get("port", 5432),
    dbname=s["rawg-db"],
    user=s["postgres"],
    password=s["password"],
    connect_timeout=10,
    sslmode="require"   # en RDS suele ir bien
)
print("OK conectado")
conn.close()


KeyError: 'rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com'

In [12]:

# === 1: CONFIGURACIÓN DE CONEXIÓN ===


REGION = "eu-north-1"
SECRET_NAME = "arn:aws:secretsmanager:eu-north-1:637910750724:secret:Postgre-gGI3hV"   # o el ARN

def get_db_secret(secret_name=SECRET_NAME, region=REGION):
    client = boto3.client("secretsmanager", region_name=region)
    resp = client.get_secret_value(SecretId=secret_name)
    secret_str = resp["SecretString"]
    return json.loads(secret_str)

def connect_postgres():
    # Crea conexión a PostgreSQL
    try:
        s = get_db_secret()
        conn = psycopg2.connect(
            host=s["rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com"],
            port=s.get("port", 5432),
            dbname=s["rawg-db"],
            user=s["postgres"],
            password=s["password"],
            connect_timeout=10
            )
        print("Conexión establecida con PostgreSQL")
        return conn
    except Exception as e:
        print(f"Error al conectar: {e}")
        return None

# test rápido
conn = connect_postgres()
cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())
cur.close()
conn.close()


Error al conectar: 'rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com'


AttributeError: 'NoneType' object has no attribute 'cursor'

In [ ]:
# ============================================
# PASO 1: CONFIGURACIÓN DE CONEXIÓN
# ============================================

# Configuración de conexión a PostgreSQL
DB_CONFIG = {
    'host': 'rawg-db.cdeis4oi2b0g.eu-north-1.rds.amazonaws.com',  # Cambiar al RDS endpoint en AWS o localhost
    'database': 'rawg_db',
    'user': 'postgres',
    'password': 'tu_password',
    'port': 5432
}

import boto3
from botocore.exceptions import ClientError


def crear_conexion():
    """Crea conexión a PostgreSQL"""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print("Conexión establecida con PostgreSQL")
        return conn
    except Exception as e:
        print(f"Error al conectar: {e}")
        return None

# Probar conexión
conn = crear_conexion()
if conn:
    cursor = conn.cursor()
    cursor.execute("SELECT version();")
    version = cursor.fetchone()
    print(f"PostgreSQL version: {version[0]}")
    cursor.close()
    conn.close()

# ============================================
# PASO 2: CREAR ESQUEMA DE BASE DE DATOS
# ============================================

def crear_esquema_completo():
    """Crea todas las tablas necesarias en PostgreSQL"""
    
    conn = crear_conexion()
    if not conn:
        return False
    
    cursor = conn.cursor()
    
    print("Creando esquema de base de datos...")
    print("=" * 60)
    
    # SQL para crear tablas (el esquema completo de arriba)
    schema_sql = """
    -- Eliminar tablas si existen (orden inverso por foreign keys)
    DROP TABLE IF EXISTS game_tags CASCADE;
    DROP TABLE IF EXISTS tags CASCADE;
    DROP TABLE IF EXISTS game_stores CASCADE;
    DROP TABLE IF EXISTS stores CASCADE;
    DROP TABLE IF EXISTS game_genres CASCADE;
    DROP TABLE IF EXISTS genres CASCADE;
    DROP TABLE IF EXISTS game_platforms CASCADE;
    DROP TABLE IF EXISTS platforms CASCADE;
    DROP TABLE IF EXISTS ratings_distribution CASCADE;
    DROP TABLE IF EXISTS esrb_games CASCADE;
    DROP TABLE IF EXISTS esrb_ratings CASCADE;
    DROP TABLE IF EXISTS games_status CASCADE;
    DROP TABLE IF EXISTS games CASCADE;
    
    -- TABLA PRINCIPAL: games
    CREATE TABLE games (
        game_id INTEGER PRIMARY KEY,
        game_name VARCHAR(500) NOT NULL,
        tba VARCHAR(10),
        released_ym VARCHAR(7),
        updated_ym VARCHAR(7),
        game_rating NUMERIC(5,2),
        raitings_count INTEGER,
        game_added INTEGER,
        playtime INTEGER,
        suggestions_count INTEGER,
        esrb_id INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    
    -- TABLA: games_status
    CREATE TABLE games_status (
        game_id INTEGER PRIMARY KEY,
        yet INTEGER DEFAULT 0,
        owned INTEGER DEFAULT 0,
        beaten INTEGER DEFAULT 0,
        toplay INTEGER DEFAULT 0,
        dropped INTEGER DEFAULT 0,
        playing INTEGER DEFAULT 0,
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE
    );
    
    -- CATÁLOGO: esrb_ratings
    CREATE TABLE esrb_ratings (
        esrb_id INTEGER PRIMARY KEY,
        esrb_name VARCHAR(50) NOT NULL,
        esrb_slug VARCHAR(50)
    );
    
    -- TABLA: esrb_games
    CREATE TABLE esrb_games (
        game_id INTEGER PRIMARY KEY,
        esrb_id INTEGER NOT NULL,
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        FOREIGN KEY (esrb_id) REFERENCES esrb_ratings(esrb_id)
    );
    
    -- TABLA: ratings_distribution
    CREATE TABLE ratings_distribution (
        id SERIAL PRIMARY KEY,
        game_id INTEGER NOT NULL,
        title VARCHAR(50) NOT NULL,
        count INTEGER DEFAULT 0,
        percent NUMERIC(5,2) DEFAULT 0.0,
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        UNIQUE(game_id, title)
    );
    
    -- CATÁLOGO: platforms
    CREATE TABLE platforms (
        platform_id INTEGER PRIMARY KEY,
        platform_name VARCHAR(200) NOT NULL
    );
    
    -- TABLA: game_platforms
    CREATE TABLE game_platforms (
        id SERIAL PRIMARY KEY,
        game_id INTEGER NOT NULL,
        platform_id INTEGER NOT NULL,
        platform_name VARCHAR(200),
        released_at VARCHAR(7),
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        FOREIGN KEY (platform_id) REFERENCES platforms(platform_id),
        UNIQUE(game_id, platform_id)
    );
    
    -- CATÁLOGO: genres
    CREATE TABLE genres (
        genre_id INTEGER PRIMARY KEY,
        genre_name VARCHAR(100) NOT NULL
    );
    
    -- TABLA: game_genres
    CREATE TABLE game_genres (
        id SERIAL PRIMARY KEY,
        game_id INTEGER NOT NULL,
        genre_id INTEGER NOT NULL,
        genre_name VARCHAR(100),
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id),
        UNIQUE(game_id, genre_id)
    );
    
    -- CATÁLOGO: stores
    CREATE TABLE stores (
        store_id INTEGER PRIMARY KEY,
        store_name VARCHAR(200) NOT NULL
    );
    
    -- TABLA: game_stores
    CREATE TABLE game_stores (
        id SERIAL PRIMARY KEY,
        game_id INTEGER NOT NULL,
        store_id INTEGER NOT NULL,
        store_name VARCHAR(200),
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        FOREIGN KEY (store_id) REFERENCES stores(store_id),
        UNIQUE(game_id, store_id)
    );
    
    -- CATÁLOGO: tags
    CREATE TABLE tags (
        tag_id INTEGER PRIMARY KEY,
        tag_name VARCHAR(200) NOT NULL
    );
    
    -- TABLA: game_tags
    CREATE TABLE game_tags (
        id SERIAL PRIMARY KEY,
        game_id INTEGER NOT NULL,
        tag_id INTEGER NOT NULL,
        tag_name VARCHAR(200),
        FOREIGN KEY (game_id) REFERENCES games(game_id) ON DELETE CASCADE,
        FOREIGN KEY (tag_id) REFERENCES tags(tag_id),
        UNIQUE(game_id, tag_id)
    );
    
    -- ÍNDICES
    CREATE INDEX idx_games_released ON games(released_ym);
    CREATE INDEX idx_games_rating ON games(game_rating);
    CREATE INDEX idx_game_platforms_game ON game_platforms(game_id);
    CREATE INDEX idx_game_genres_game ON game_genres(game_id);
    CREATE INDEX idx_game_stores_game ON game_stores(game_id);
    CREATE INDEX idx_game_tags_game ON game_tags(game_id);
    """
    
    try:
        cursor.execute(schema_sql)
        conn.commit()
        print("✅ Esquema creado exitosamente")
        
        # Verificar tablas creadas
        cursor.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        
        tablas = cursor.fetchall()
        print(f"\n📊 Tablas creadas ({len(tablas)}):")
        for tabla in tablas:
            print(f"   • {tabla[0]}")
        
        cursor.close()
        conn.close()
        return True
        
    except Exception as e:
        print(f"❌ Error al crear esquema: {e}")
        conn.rollback()
        cursor.close()
        conn.close()
        return False

# Crear esquema
crear_esquema_completo()

# ============================================
# PASO 3: FUNCIONES DE CARGA
# ============================================

def cargar_dataframe_a_tabla(df, tabla, conn, usar_copy=True):
    """
    Carga un DataFrame a una tabla de PostgreSQL
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame a cargar
    tabla : str
        Nombre de la tabla destino
    conn : connection
        Conexión a PostgreSQL
    usar_copy : bool
        Si True, usa COPY (más rápido). Si False, usa INSERT
    """
    
    if df.empty:
        print(f"   ⚠️ {tabla}: DataFrame vacío, omitiendo...")
        return 0
    
    cursor = conn.cursor()
    
    try:
        if usar_copy:
            # Método COPY (más rápido para grandes volúmenes)
            from io import StringIO
            
            # Preparar datos
            buffer = StringIO()
            df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
            buffer.seek(0)
            
            # COPY
            columnas = ', '.join(df.columns)
            cursor.copy_from(buffer, tabla, sep='\t', null='\\N', columns=df.columns.tolist())
            
        else:
            # Método INSERT (más lento pero más compatible)
            columnas = ', '.join(df.columns)
            placeholders = ', '.join(['%s'] * len(df.columns))
            insert_query = f"INSERT INTO {tabla} ({columnas}) VALUES ({placeholders})"
            
            # Convertir DataFrame a lista de tuplas
            datos = [tuple(x) for x in df.to_numpy()]
            
            execute_batch(cursor, insert_query, datos, page_size=1000)
        
        conn.commit()
        print(f"   ✅ {tabla}: {len(df)} filas cargadas")
        return len(df)
        
    except Exception as e:
        conn.rollback()
        print(f"   ❌ Error en {tabla}: {e}")
        return 0
    finally:
        cursor.close()

# ============================================
# PASO 4: FUNCIÓN PRINCIPAL DE CARGA
# ============================================

def cargar_datos_completo(resultado_transformacion, usar_copy=True):
    """
    Carga todos los datos transformados a PostgreSQL
    
    Parameters:
    -----------
    resultado_transformacion : dict
        Resultado de transformar_datos_completo()
    usar_copy : bool
        Usar método COPY (más rápido)
    
    Returns:
    --------
    dict : Estadísticas de carga
    """
    
    print("\n" + "=" * 60)
    print("INICIANDO CARGA A POSTGRESQL")
    print("=" * 60)
    
    conn = crear_conexion()
    if not conn:
        return None
    
    estadisticas = {
        'inicio': datetime.now(),
        'tablas_cargadas': 0,
        'filas_totales': 0,
        'detalles': {}
    }
    
    # Orden de carga (respetando foreign keys)
    orden_carga = [
        # 1. Tabla principal
        ('games', 'games'),
        
        # 2. Catálogos independientes
        ('platforms', 'catalogs.platforms'),
        ('genres', 'catalogs.genres'),
        ('stores', 'catalogs.stores'),
        ('tags', 'catalogs.tags'),
        ('esrb_ratings', 'catalogs.esrb'),  # Si existe
        
        # 3. Tablas dependientes
        ('games_status', 'games_status'),
        ('esrb_games', 'esrb_games'),
        ('ratings_distribution', 'ratings_distribution'),
        ('game_platforms', 'game_platforms'),
        ('game_genres', 'game_genres'),
        ('game_stores', 'game_stores'),
        ('game_tags', 'game_tags'),
    ]
    
    for tabla_sql, ruta_df in orden_carga:
        print(f"\n📌 Cargando {tabla_sql}...")
        
        # Obtener DataFrame
        if '.' in ruta_df:
            # Es un catálogo
            partes = ruta_df.split('.')
            if partes[0] in resultado_transformacion and partes[1] in resultado_transformacion[partes[0]]:
                df = resultado_transformacion[partes[0]][partes[1]]
            else:
                print(f"   ⚠️ {tabla_sql}: No encontrado en resultado")
                continue
        else:
            # Es una tabla principal
            if ruta_df in resultado_transformacion:
                df = resultado_transformacion[ruta_df]
            else:
                print(f"   ⚠️ {tabla_sql}: No encontrado en resultado")
                continue
        
        # Cargar
        filas = cargar_dataframe_a_tabla(df, tabla_sql, conn, usar_copy)
        
        if filas > 0:
            estadisticas['tablas_cargadas'] += 1
            estadisticas['filas_totales'] += filas
            estadisticas['detalles'][tabla_sql] = filas
    
    conn.close()
    
    estadisticas['fin'] = datetime.now()
    estadisticas['duracion'] = (estadisticas['fin'] - estadisticas['inicio']).total_seconds()
    
    # Reporte final
    print("\n" + "=" * 60)
    print("✅ CARGA COMPLETADA")
    print("=" * 60)
    print(f"\n📊 Estadísticas:")
    print(f"   • Tablas cargadas: {estadisticas['tablas_cargadas']}")
    print(f"   • Filas totales: {estadisticas['filas_totales']}")
    print(f"   • Duración: {estadisticas['duracion']:.2f} segundos")
    
    print(f"\n📋 Detalle por tabla:")
    for tabla, filas in estadisticas['detalles'].items():
        print(f"   • {tabla}: {filas} filas")
    
    return estadisticas

# ============================================
# PASO 5: VALIDACIÓN DE CARGA
# ============================================

def validar_carga():
    """Valida que los datos se hayan cargado correctamente"""
    
    print("\n" + "=" * 60)
    print("🔍 VALIDANDO CARGA")
    print("=" * 60)
    
    conn = crear_conexion()
    if not conn:
        return False
    
    cursor = conn.cursor()
    
    # Contar registros en cada tabla
    tablas = [
        'games', 'games_status', 'esrb_games', 'ratings_distribution',
        'game_platforms', 'game_genres', 'game_stores', 'game_tags',
        'platforms', 'genres', 'stores', 'tags'
    ]
    
    print(f"\n📊 Registros por tabla:")
    for tabla in tablas:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {tabla};")
            count = cursor.fetchone()[0]
            print(f"   • {tabla}: {count} registros")
        except Exception as e:
            print(f"   ❌ {tabla}: Error - {e}")
    
    # Validar integridad referencial
    print(f"\n🔗 Validando integridad referencial:")
    
    # Verificar que todos los game_id en tablas secundarias existen en games
    validaciones = [
        ("games_status", "SELECT COUNT(*) FROM games_status WHERE game_id NOT IN (SELECT game_id FROM games)"),
        ("game_platforms", "SELECT COUNT(*) FROM game_platforms WHERE game_id NOT IN (SELECT game_id FROM games)"),
        ("game_genres", "SELECT COUNT(*) FROM game_genres WHERE game_id NOT IN (SELECT game_id FROM games)"),
    ]
    
    for tabla, query in validaciones:
        cursor.execute(query)
        huerfanos = cursor.fetchone()[0]
        if huerfanos == 0:
            print(f"   ✅ {tabla}: Integridad OK")
        else:
            print(f"   ⚠️ {tabla}: {huerfanos} registros huérfanos")
    
    cursor.close()
    conn.close()
    
    print("\n✅ Validación completada")
    return True

# ============================================
# PASO 6: CONSULTAS DE EJEMPLO
# ============================================

def ejecutar_consultas_ejemplo():
    """Ejecuta algunas consultas de ejemplo para verificar"""
    
    print("\n" + "=" * 60)
    print("🔍 CONSULTAS DE EJEMPLO")
    print("=" * 60)
    
    conn = crear_conexion()
    if not conn:
        return
    
    # Consulta 1: Top 10 juegos mejor valorados
    print(f"\n📊 Top 10 juegos mejor valorados:")
    query1 = """
        SELECT game_name, game_rating, raitings_count
        FROM games
        WHERE game_rating > 0
        ORDER BY game_rating DESC, raitings_count DESC
        LIMIT 10;
    """
    df1 = pd.read_sql(query1, conn)
    print(df1)
    
    # Consulta 2: Juegos por género
    print(f"\n📊 Cantidad de juegos por género:")
    query2 = """
        SELECT g.genre_name, COUNT(*) as cantidad
        FROM game_genres gg
        JOIN genres g ON gg.genre_id = g.genre_id
        GROUP BY g.genre_name
        ORDER BY cantidad DESC;
    """
    df2 = pd.read_sql(query2, conn)
    print(df2)
    
    # Consulta 3: Juegos por plataforma
    print(f"\n📊 Top 10 plataformas con más juegos:")
    query3 = """
        SELECT p.platform_name, COUNT(*) as cantidad
        FROM game_platforms gp
        JOIN platforms p ON gp.platform_id = p.platform_id
        GROUP BY p.platform_name
        ORDER BY cantidad DESC
        LIMIT 10;
    """
    df3 = pd.read_sql(query3, conn)
    print(df3)
    
    conn.close()